# 04 · Structural conditions and overdose rates

Notebook 03 showed *who* lives in the tracts where treatment supply falls short of need. This notebook asks a related question with a regression: across all Cook County tracts, which structural conditions go along with higher overdose death rates when they're considered together?

**What this model is not:** it doesn't estimate the effect of treatment access. The need-based access score has deaths in its denominator, so it can't also be a predictor of deaths, and clinics open where deaths are already high (reverse causality). The results are descriptive associations, not causes. See `docs/methodology.md`, D6.

**Framing:** race and ethnicity variables stand in for the structural conditions that follow residential segregation in Chicago (disinvestment, concentrated drug markets, policing, and access to care), not for anything about individuals or groups.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from src import model, plots
from src.config import PROCESSED_DIR, REFERENCE_DIR

plots.set_style()

In [ ]:
tract_table = pd.read_csv(PROCESSED_DIR / "tract_table.csv", dtype={"GEOID": str})
data = model.prepare_model_data(tract_table)
print(f"{len(data):,} tracts in the model (population 500+ with complete ACS data)")
print(model.model_formula())

## Why a count model, and why negative binomial

The outcome is a count of deaths per tract: whole numbers, never negative, and mostly small. Linear regression assumes a continuous, roughly normal outcome with constant variance, which a count doesn't have, and it can predict negative deaths. Poisson regression is built for counts, and the population offset turns counts into rates.

Poisson assumes the variance equals the mean. If the variance is much larger (overdispersion), Poisson standard errors come out too small and everything looks more certain than it is. The dispersion ratio checks for that.

In [ ]:
poisson, dispersion = model.fit_poisson(data)
print(f"Poisson dispersion ratio: {dispersion:.2f} (1.0 = no overdispersion)")
print(f"Mean deaths per tract: {data['overdose_deaths'].mean():.1f}, variance: {data['overdose_deaths'].var():.1f}")

A ratio well above 1 means the negative binomial is the right choice: it adds a dispersion parameter (alpha) so the variance can grow faster than the mean.

## Checking for spatial clustering

Neighboring tracts share a lot (the same drug markets, the same disinvestment), so the model's errors are probably correlated in space. If they are, the observations aren't independent and even the negative binomial standard errors are too small. Moran's I on the residuals tests for this.

In [ ]:
negative_binomial, alpha = model.fit_negative_binomial(data)
print(f"Estimated alpha: {alpha:.3f}")
print(f"AIC  Poisson: {poisson.aic:,.0f}   Negative binomial: {negative_binomial.aic:,.0f}  (lower is better)")

tracts = gpd.read_file(REFERENCE_DIR / "cook_tracts.gpkg").set_index("GEOID").loc[data["GEOID"]]
moran, p_value = model.morans_i(negative_binomial.resid_pearson, tracts.geometry)
print(f"Moran's I of residuals: {moran:.3f} (permutation p = {p_value:.3f})")

The residuals are clearly clustered, so the fitted model uses **cluster-robust standard errors**: tracts are grouped by the first two digits of their tract code, which groups neighbors together (in Chicago, those digits line up with community areas). This widens the confidence intervals to account for tracts in the same area not being independent.

## Results

In [ ]:
rate_ratios = model.rate_ratio_table(negative_binomial)
rate_ratios.round(3)

In [ ]:
table = rate_ratios.sort_values("rate_ratio")
fig, ax = plt.subplots(figsize=(8, 4))
positions = range(len(table))

ax.hlines(positions, table["ci_low"], table["ci_high"], color=plots.BLUE, linewidth=2)
ax.plot(table["rate_ratio"], positions, "o", color=plots.BLUE, markersize=8,
        markeredgecolor=plots.SURFACE, markeredgewidth=1.5)
ax.axvline(1, color=plots.TEXT_SECONDARY, linewidth=1, linestyle="--")
# label the reference line at the bottom, clear of the title
ax.set_ylim(-0.9, len(table) - 0.5)
ax.text(1.002, -0.75, "1.0 = no association", fontsize=8, color=plots.TEXT_SECONDARY, va="center")

ax.set_yticks(list(positions), table.index)
ax.set_xlabel("Rate ratio (with 95% confidence interval)")
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.set_title("Structural conditions and tract overdose death rates")
plots.add_source_note(fig, "Negative binomial model of 2015-2025 overdose deaths with a population offset, "
                           f"{len(data):,} tracts. 95% CIs use standard errors clustered by area.")
plots.save_figure(fig, "model_rate_ratios.png")
plt.show()

## A robustness check

Swap poverty for median household income (they measure similar things, so the model only uses one at a time) and confirm the other rate ratios don't move much.

In [ ]:
income_data = data.dropna(subset=["median_household_income"]).copy()
income_data["log_income"] = np.log(income_data["median_household_income"])
income_formula = model.model_formula().replace("pct_poverty_per10", "log_income")

income_model = smf.glm(income_formula, income_data, family=sm.families.NegativeBinomial(alpha=alpha),
                       offset=income_data["log_exposure"])
income_result = income_model.fit(cov_type="cluster",
                                 cov_kwds={"groups": pd.factorize(income_data["cluster"])[0]})

pd.DataFrame({
    "main_model": np.exp(negative_binomial.params),
    "income_instead_of_poverty": np.exp(income_result.params),
}).drop(index="Intercept").round(3)

## What this shows

- **Poisson was the wrong tool, and the check caught it.** The variance of tract deaths is about 15 times the mean (177.5 vs 11.5), and the Poisson dispersion ratio is 6.8. The negative binomial fits far better (AIC 8,373 vs 12,149).
- **Residuals cluster in space** (Moran's I = 0.30, p = 0.001), so the confidence intervals use standard errors clustered by area. Without that correction they would be too narrow.
- **Vehicle access has the strongest association.** Each 10 more points of car-free households goes with a 20% higher overdose death rate (95% CI 12% to 29%), holding poverty, insurance, and racial composition fixed. That ties directly to why transit travel time is the next access measure to build.
- **Poverty and insurance matter independently.** +10 points of poverty: 13% higher rate (7% to 19%). +10 points uninsured: 15% higher (4% to 27%).
- **Racial composition still matters after these controls.** +10 points Black residents: 15% higher rate (10% to 21%). The model can't say why, and it would be wrong to read this as being about individuals. It most likely reflects conditions tied to segregation that ACS doesn't measure: disinvestment, concentrated drug markets, policing, and a history of unequal access to care.
- **Treatment per resident isn't significantly associated with death rates** once structure is accounted for (rate ratio 1.04, CI 0.99 to 1.09). The slight positive lean fits clinics having opened where need was already high.
- **Robust to the income measure.** Swapping poverty for log median income leaves the other rate ratios nearly unchanged. Doubling a tract's median income goes with roughly a 28% lower rate.

**Limits:** cross-sectional and ecological (tract-level patterns don't describe individuals), deaths are counted where they happen, not where people live, and ACS estimates carry sizable margins of error in small tracts.